In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('olist_processed_sample.csv')

df_orders = pd.read_csv('../data base/olist_orders_dataset.csv')
df_items = pd.read_csv('../data base/olist_order_items_dataset.csv')
df_products = pd.read_csv('../data base/olist_products_dataset.csv')
df_customers = pd.read_csv('../data base/olist_customers_dataset.csv')
df_reviews = pd.read_csv('../data base/olist_order_reviews_dataset.csv')
#print(dfCleaned['price'].sum())

#print("Numero de datos que tenemos",df.shape)   # (1000 , 10)#
#print("columnas que tenemos",df.columns) #columnas que tenemos ['order_id', 'customer_id', 'customer_state', 'product_category','price', 'freight_value', 'order_purchase_timestamp', 'review_score','total_order_cost', 'purchase_month']
#print("Tipos de datos",df.dtypes)
#print(df.isnull().sum()) # para ver si hay valores nulos en el dataset
#print(df['product_category'].unique()) # para ver los valores unicos de la columna review_score
#print(df[df['price'] > 220][['price', 'order_id', 'product_category']])
#correcion de campos nullos o sin tipo de dato
df['product_category'] = df['product_category'].fillna('Sin categoria')
mediana_review = df['review_score'].median()
df['review_score'] = df['review_score'].fillna(mediana_review)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])


#creacion de variables  y uso de funciones en panda
df['total_order_cost'] = df['price'] + df['freight_value']
df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['satisfaction_level'] = np.where(df['review_score'] >= 4, 'Satisfecho', 'Insatisfecho')
categori = df[df['customer_id'] == 'CUST_1098']
index_max = categori['price'].idxmax()

#print(df.loc[index_max]) 

# Resumen por categoría de producto y uso de funcion agg
category_summary = df.groupby('product_category').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('price', 'sum'),
    avg_ticket=('price', 'mean'),
    avg_satisfaction=('review_score', 'mean')
).reset_index().sort_values(by='total_revenue', ascending=False)


#limpieza y analisis de un segundo reporte con datos de las ordenes para su exportacion a power bi


df_orders_sub = df_orders[[
    'order_id', 
    'customer_id', 
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]]


df_items_sub = df_items[['order_id', 'product_id', 'price']]
df_customers_sub = df_customers[['customer_id', 'customer_state']]
df_reviews_sub = df_reviews[['order_id', 'review_score']]


df_merged = df_orders_sub.merge(df_items_sub, on='order_id', how='left') \
                         .merge(df_customers_sub, on='customer_id', how='left') \
                         .merge(df_reviews_sub, on='order_id', how='left')


 # Limpieza rápida: Eliminar duplicados exactos si los hay
df_merged = df_merged.drop_duplicates()

# Convertir columnas de fecha a formato datetime
fechas = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in fechas:
    df_merged[col] = pd.to_datetime(df_merged[col], errors='coerce')

df_merged.to_csv('olist_powerbi_optimizado.csv', index=False, encoding='utf-8')









Index(['order_id', 'customer_id', 'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'product_id', 'price', 'customer_state', 'review_score'],
      dtype='str')
Index(['order_id', 'customer_id', 'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'product_id', 'price', 'customer_state', 'review_score'],
      dtype='str')
Index(['order_id', 'customer_id', 'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'product_id', 'price', 'customer_state', 'review_score'],
      dtype='str')
